In [ ]:
import pandas as pd
import scanpy as sc
import bbknn as bk
import numpy as np

In [ ]:
adata1=sc.read_csv('D01_cellphonedb/D01sigmeans.csv',first_column_names=bool,delimiter=',').T
adata2=sc.read_csv('D03_cellphonedb/D03sigmeans.csv',first_column_names=bool,delimiter=',').T
adata3=sc.read_csv('D07_cellphonedb/D07sigmeans.csv',first_column_names=bool,delimiter=',').T
adata4=sc.read_csv('D30_cellphonedb/D30sigmeans.csv',first_column_names=bool,delimiter=',').T
adata5=sc.read_csv('Unop_cellphonedb/Unopsigmeans.csv',first_column_names=bool,delimiter=',').T
adata6=sc.read_csv('Sham_cellphonedb/Shamsigmeans.csv',first_column_names=bool,delimiter=',').T

cpdb_adata=adata1.concatenate(adata2,adata3,adata4,adata5,adata6)

In [ ]:
cellp = 'data/cpdb_adata.h5ad'
cpdb_adata.write(cellp)
cpdb_adata.write_csvs(cellp[:-5],)

In [ ]:
df = pd.read_csv('cpdb_adata/obs_edited.csv')
df

In [ ]:
import re
    
def split_column(cell_interaction_rows):
    
    cell_interaction_dict = {}
    
    cell_interactions_split = cell_interaction_rows.str.split('|')
    receivers = cell_interactions_split.apply(lambda x: x[0])
    cell_interaction_dict['Rcluster_MI'] = receivers
    receiver_types = receivers.str.replace('_.*$', '')
    cell_interaction_dict['Rcluster'] = receiver_types
    receiver_cells = receiver_types.str.replace('\d+', '')
    cell_interaction_dict['RC_type'] = receiver_cells
    senders = cell_interactions_split.apply(lambda x: x[1])
    cell_interaction_dict['Scluster_MI'] = senders
    sender_types = senders.str.replace('_.*$', '')
    cell_interaction_dict['Scluster'] = sender_types
    sender_cells = sender_types.str.replace('\d+', '')
    cell_interaction_dict['SC_type'] = sender_cells
    timepoint = receivers.str.replace('^.*_', '')
    cell_interaction_dict['MI'] = timepoint
    receiver_cell_MI = receiver_cells.str.cat(others = '_' + timepoint)
    cell_interaction_dict['RC_type_MI'] = receiver_cell_MI
    sender_cell_MI = sender_cells.str.cat(others = '_' + timepoint)
    cell_interaction_dict['SC_type_MI'] = sender_cell_MI
    cluster_interactions = receiver_types.str.cat(others = '|' + sender_types)
    cell_interaction_dict['Cluster_interactions'] = cluster_interactions
    cell_interactions = receiver_cells.str.cat(others = '|' + sender_cells)
    cell_interaction_dict['Cell_type_interactions'] = cell_interactions 
    cell_timepoint_interactions = receiver_cell_MI.str.cat(others = '|' + sender_cell_MI)
    cell_interaction_dict['Cell_type_MI_interactions'] = cell_timepoint_interactions
    
    return(pd.DataFrame(cell_interaction_dict))

In [ ]:
df2 = split_column(df['Interactions'])
df2.to_csv('obs_edited.csv')

In [ ]:
cpdb_adata.obs = df2

In [ ]:
sc.tl.pca(cpdb_adata,svd_solver='arpack')

In [ ]:
fibro_macrophage_interactions = cpdb_adata[(cpdb_adata.obs['Cell_type_interactions'] == 'F|M') | (cpdb_adata.obs['Cell_type_interactions'] == 'M|F')]
fibro_macrophage_interactions

In [ ]:
macro_to_fibro = cpdb_adata[(cpdb_adata.obs['Cell_type_interactions'] == 'F|M')]

In [ ]:
fibro_to_macro = cpdb_adata[(cpdb_adata.obs['Cell_type_interactions'] == 'M|F')]

In [ ]:
fibro_endothelial_interactions = cpdb_adata[(cpdb_adata.obs['Cell_type_interactions'] == 'F|E') | (cpdb_adata.obs['Cell_type_interactions'] == 'E|F')]

In [ ]:
endo_to_fibro = cpdb_adata[(cpdb_adata.obs['Cell_type_interactions'] == 'F|E')]

In [ ]:
fibro_to_endo = cpdb_adata[(cpdb_adata.obs['Cell_type_interactions'] == 'E|F')]

In [ ]:
fibro_cm_interactions = cpdb_adata[(cpdb_adata.obs['Cell_type_interactions'] == 'F|CM') | (cpdb_adata.obs['Cell_type_interactions'] == 'CM|F')]

In [ ]:
fibro_macro_interactions_D01 = cpdb_adata[(cpdb_adata.obs['Cell_type_MI_interactions'] == 'F_D01|M_D01') | (cpdb_adata.obs['Cell_type_MI_interactions'] == 'M_D01|F_D01')]

In [ ]:
fibro_macro_interactions_D03 = cpdb_adata[(cpdb_adata.obs['Cell_type_MI_interactions'] == 'F_D03|M_D03') | (cpdb_adata.obs['Cell_type_MI_interactions'] == 'M_D03|F_D03')]

In [ ]:
fibro_macro_interactions_D07 = cpdb_adata[(cpdb_adata.obs['Cell_type_MI_interactions'] == 'F_D07|M_D07') | (cpdb_adata.obs['Cell_type_MI_interactions'] == 'M_D07|F_D07')]

In [ ]:
fibro_macro_interactions_D30 = cpdb_adata[(cpdb_adata.obs['Cell_type_MI_interactions'] == 'F_D30|M_D30') | (cpdb_adata.obs['Cell_type_MI_interactions'] == 'M_D30|F_D30')]

In [ ]:
fibro_macro_interactions_Sham = cpdb_adata[(cpdb_adata.obs['Cell_type_MI_interactions'] == 'F_Sham|M_Sham') | (cpdb_adata.obs['Cell_type_MI_interactions'] == 'M_Sham|F_Sham')]

In [ ]:
fibro_macro_interactions_Unop = cpdb_adata[(cpdb_adata.obs['Cell_type_MI_interactions'] == 'F_Unop|M_Unop') | (cpdb_adata.obs['Cell_type_MI_interactions'] == 'M_Unop|F_Unop')]

In [ ]:
sc.tl.rank_genes_groups(cpdb_adata,groupby='Cell_type_interactions',n_genes=10,method='wilcoxon',corr_method='bonferroni')

In [ ]:
sc.tl.dendrogram(cpdb_adata,groupby='Cell_type_interactions')

In [ ]:
sc.settings.set_figure_params(dpi=300,fontsize=7)

In [ ]:
sc.pl.rank_genes_groups_dotplot(cpdb_adata,n_genes=10,show=False) #map receptors on umap

In [ ]:
sc.tl.rank_genes_groups(cpdb_adata, 'MI', method = 'wilcoxon', n_genes = 500, use_raw = True)
result = cpdb_adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
markers = pd.DataFrame(
{group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'pvals_adj', 'logfoldchanges','scores']})
markers.head(10)

In [ ]:
sc.tl.rank_genes_groups(cpdb_adata, 'Cell_type_MI_interactions', method = 'wilcoxon', n_genes = 500, use_raw = True)
result = cpdb_adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
markers = pd.DataFrame(
{group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'pvals_adj', 'logfoldchanges','scores']})
markers.head(10)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
import numpy as np

In [ ]:
sc.settings.set_figure_params(dpi=300,fontsize=14)
fibro_macro_top_ints = markers['F|M_n'][0:10].tolist()
macro_fibro_top_ints = markers['M|F_n'][0:10].tolist()
fibro_macro_markers = fibro_macro_top_ints + macro_fibro_top_ints
fibro_macrophage_interactions.obs_names = fibro_macrophage_interactions.obs_names.astype(str)
dp = sc.pl.dotplot(fibro_macrophage_interactions, fibro_macro_markers, groupby='Cell_type_interactions',color_map='Blues')

In [ ]:
markers.to_csv('data/cell_type_MI_int_dge.csv')

In [ ]:
old_names = fibro_macrophage_interactions.obs_names 
fibro_macrophage_interactions.obs_names = fibro_macrophage_interactions.obs_names.astype(str)